# Week 4 - Generator pipeline, baseline, and eval harness

Picking up from wk3_build_retriever_v2.ipynb, which left a working retrieve() over the 10,337 chunk
index. This week: feed what it retrieves into a local LLM with a strict grounding prompt, run a
no-RAG baseline alongside it, and build the measuring kit that week 6 will reuse.

Two models on purpose: Mistral generates, Llama judges. An LLM grading its own output scores
itself too kindly - there is self preferecne bias (Panickssery et al 2024), so the
judge has to be a different model.

In [1]:
import json
import re
from collections import Counter
from datetime import datetime, timezone

import chromadb
import ollama
import pandas as pd
import textstat
from sentence_transformers import SentenceTransformer

/Users/charlesbruce/Desktop/UWE/venv-arm64/lib/python3.13/site-packages/textstat/textstat.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/charlesbruce/Desktop/UWE/venv-arm64/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# config in one place,everything here gets written into the run log so results stay traceable.
GENERATOR_MODEL = 'mistral:7b-instruct'
JUDGE_MODEL = 'llama3.1:8b'
TEMPERATURE = 0          # reproducibility

DB_DIR = 'chroma_db'
COLLECTION = 'medlineplus_english'
EMBED_MODEL = 'BAAI/bge-small-en-v1.5'
TOP_K = 5
MAX_PER_TOPIC = 2

RUN_LOG = 'wk4_runs.jsonl'

In [3]:
# record the exact model builds - quantized models differ between versions and
# this needs to go in the report's reproducibility notes
for m in [GENERATOR_MODEL, JUDGE_MODEL]:
    info = ollama.show(m)
    details = info.get('details', {})
    print(m, '|', details.get('parameter_size'), '|', details.get('quantization_level'))

mistral:7b-instruct | 7.2B | Q4_K_M
llama3.1:8b | 8.0B | Q4_K_M


## 1 - is the generator working and fast enough

Not muchhere, just proving the thing runs before building on top of it. If this cell takes
more than ~30s per answer the whole week's plan needs rethinking, so it's the first risk to check.

In [4]:
import time

def generate(prompt, model=GENERATOR_MODEL):
    resp = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': TEMPERATURE},
    )
    return resp['message']['content'].strip()


t0 = time.time()
out = generate('In one sentence, what is blood pressure?')
print(f"({time.time()-t0:.1f}s) {out}")

(13.3s) Blood pressure is the force exerted by circulating blood on the walls of the arteries.


## Reconnect the retriever

Same retrieve() as week 3, condensed. Talks to the chroma_db folder the retriever notebook built 

In [5]:
embedder = SentenceTransformer(EMBED_MODEL)
coll = chromadb.PersistentClient(path=DB_DIR).get_collection(COLLECTION)
print('index loaded:', coll.count(), 'chunks')


def retrieve(query, k=TOP_K, max_per_topic=MAX_PER_TOPIC):
    q_emb = embedder.encode('Represent this sentence for searching relevant passages: ' + query)
    res = coll.query(query_embeddings=[q_emb.tolist()], n_results=k * 10)

    hits, spillover, per_topic = [], [], Counter()
    for doc, meta, dist in zip(res['documents'][0], res['metadatas'][0], res['distances'][0]):
        hit = {'text': doc, 'title': meta['title'], 'url': meta['url'],
               'group': meta['group'], 'similarity': round(1 - dist, 3)}
        if per_topic[meta['topic_id']] >= max_per_topic:
            spillover.append(hit)
            continue
        per_topic[meta['topic_id']] += 1
        hits.append(hit)
        if len(hits) == k:
            break
    if len(hits) < k:
        hits.extend(spillover[:k - len(hits)])
    return hits

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4208.62it/s]


index loaded: 10337 chunks


## 2 - grounding prompt and the RAG pipeline

The prompt is the safety control, so being strict on purpose: answer only from the passages, cite
the topic title after each claim, refuse if the passages don't cover it. Med-PaLM measured what
happens without this - roughly 30% of free-form answers from a much bigger model were rated
potentially harmful. Our system never answers from its own memory.

Citing by topic title (in square brackets) rather than numbers because titles can be checked
mechanically against what was retrieved - that becomes the citation precision metric on da 4.

In [6]:
RAG_PROMPT = """You are a patient education assistant. Answer the patient's question using ONLY the information in the passages below.

Rules:
- Use only facts stated in the passages. Do not add anything from your own knowledge.
- After each fact, cite the passage it came from by its topic title in square brackets, like [Heart Failure].
- Write in plain, everyday language a patient can understand.
- If the passages do not contain the information needed to answer, say exactly: \"I don't have reliable information to answer that. Please ask your health care provider.\"
- Do not give a diagnosis. Do not recommend changing any medicine or dose.

Passages:
{passages}

Patient's question: {question}

Answer:"""


def format_passages(hits):
    blocks = []
    for h in hits:
        blocks.append(f"Topic: {h['title']}\n{h['text']}")
    return '\n\n---\n\n'.join(blocks)


def rag_answer(question, k=TOP_K):
    hits = retrieve(question, k=k)
    prompt = RAG_PROMPT.format(passages=format_passages(hits), question=question)
    answer = generate(prompt)
    return {'question': question, 'answer': answer, 'hits': hits, 'prompt': prompt}

In [7]:
# first look - one on-corpus question, and one the corpus can't answer (refusal check)
r = rag_answer('What should I do after being discharged with heart failure?')
print('RETRIEVED:', [h['title'] for h in r['hits']])
print()
print(r['answer'])

RETRIEVED: ['Heart Attack', 'Cardiac Rehabilitation', 'Heart Failure', 'Heart Failure', 'Respiratory Failure']

After being discharged with heart failure, you should follow a cardiac rehabilitation plan. This includes exercise training, education on heart-healthy living, and counseling to reduce stress [Cardiac Rehabilitation]. You will learn how to manage your condition and make other changes such as quitting smoking, managing stress, and getting recommended physical activity [Heart Failure]. Additionally, you may need to treat any conditions that may worsen heart failure [Heart Failure]. It's important to pay close attention to your symptoms because heart failure can worsen suddenly [Heart Failure]. Always consult with your healthcare provider for personalized advice. I don't have reliable information to answer about oxygen treatment or long-term care centers, as those are related to respiratory failure, not heart failure [Respiratory Failure].


In [8]:
r = rag_answer('Can you renew my car insurance for me?')
print('RETRIEVED:', [h['title'] for h in r['hits']])
print()
print(r['answer'])

RETRIEVED: ['Health Insurance', 'Advance Directives', 'Mobility Aids', 'Mobility Aids', 'Health Insurance']

I don't have reliable information to answer that. Please ask your insurance provider about renewing your car insurance.


## 3 - no-RAG baseline and run logging

The baseline is the same model answering from its own knowledge - same tone rules, no passages, no
citation requirement (there's nothing to cite). This is the control that shows what retrieval
actually adds, same as the NRAG arm in Zhao et al. Deliberately NOT telling it to refuse - the
point of the comparison is to see what the raw model does with a medical question.

Every run from here on gets logged to JSONL with its full config, so week 6's experiments are just
re-runs with different settings and everything stays comparable.

In [9]:
BASELINE_PROMPT = """You are a patient education assistant. Answer the patient's question.

Rules:
- Write in plain, everyday language a patient can understand.
- Do not give a diagnosis. Do not recommend changing any medicine or dose.

Patient's question: {question}

Answer:"""


def baseline_answer(question):
    prompt = BASELINE_PROMPT.format(question=question)
    return {'question': question, 'answer': generate(prompt), 'hits': [], 'prompt': prompt}

In [10]:
def log_run(record, arm, scores=None, log_path=RUN_LOG):
    entry = {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'arm': arm,                                  # 'rag' or 'baseline'
        'question': record['question'],
        'answer': record['answer'],
        'retrieved': [{'title': h['title'], 'url': h['url'], 'similarity': h['similarity']}
                      for h in record['hits']],
        'scores': scores or {},
        'config': {
            'generator': GENERATOR_MODEL,
            'judge': JUDGE_MODEL,
            'temperature': TEMPERATURE,
            'k': TOP_K,
            'max_per_topic': MAX_PER_TOPIC,
            'embed_model': EMBED_MODEL,
        },
    }
    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    return entry

## 4 - the eval harness

Four metrics, cheapest first:

1. **Readability** (FKGL + Flesch Reading Ease) - pure formula, no LLM. Citations get stripped
   before scoring so bracketed titles don't distort sentence lengths.
2. **Citation precision** - what fraction of the titles the answer cites were actually in the
   retrieved passages. String matching, fully objective. Also tracking whether an answer cites
   anything at all.
3. **Faithfulness** - split the answer into sentences, ask the judge (Llama, not Mistral) whether
   each one is supported by the passages, faithfulness = fraction supported. Same shape as the
   RAGAS metric but hand-rolled so every judgment is visible and debuggable.
4. **Answer relevancy** - judge rates 1-5 whether the answer addresses the question.

Only the last two need an LLM, and it's the separate judge model both times.

In [11]:
CITE_RE = re.compile(r'\[([^\]]+)\]')


def strip_citations(text):
    return CITE_RE.sub('', text).replace('  ', ' ').strip()


def readability(answer):
    clean = strip_citations(answer)
    return {
        'fkgl': round(textstat.flesch_kincaid_grade(clean), 1),
        'fre': round(textstat.flesch_reading_ease(clean), 1),
        'words': len(clean.split()),
    }


def citation_scores(answer, hits):
    cited = [c.strip() for c in CITE_RE.findall(answer)]
    if not cited:
        return {'cited_any': False, 'citation_precision': None, 'n_citations': 0}
    retrieved_titles = {h['title'].lower() for h in hits}
    valid = sum(1 for c in cited if c.lower() in retrieved_titles)
    return {
        'cited_any': True,
        'citation_precision': round(valid / len(cited), 2),
        'n_citations': len(cited),
    }

In [12]:
def split_sentences(text):
    # citations off first, then a simple split on sentence enders. not perfect,
    # but the sentences only need to be judgeable, not linguistically pure
    clean = strip_citations(text)
    parts = re.split(r'(?<=[.!?])\s+', clean)
    return [p.strip() for p in parts if len(p.strip()) > 20]


FAITHFUL_PROMPT = """Here are passages from a medical reference, and one sentence from an answer that was written using them.

Passages:
{passages}

Sentence: \"{sentence}\"

Is the sentence fully supported by the passages? A sentence counts as supported if all its factual claims appear in the passages. General advice to consult a doctor counts as supported.

Reply with exactly one word: YES or NO."""


def judge_faithfulness(record):
    sentences = split_sentences(record['answer'])
    if not sentences:
        return {'faithfulness': None, 'n_sentences': 0, 'unsupported': []}
    passages = format_passages(record['hits'])
    verdicts, unsupported = [], []
    for s in sentences:
        out = generate(FAITHFUL_PROMPT.format(passages=passages, sentence=s), model=JUDGE_MODEL)
        ok = out.strip().upper().startswith('YES')
        verdicts.append(ok)
        if not ok:
            unsupported.append(s)
    return {
        'faithfulness': round(sum(verdicts) / len(verdicts), 2),
        'n_sentences': len(sentences),
        'unsupported': unsupported,   # kept for the failure log - these ARE the hallucination candidates
    }


RELEVANCY_PROMPT = """A patient asked: \"{question}\"

They received this answer:
\"{answer}\"

On a scale of 1 to 5, how well does the answer address the patient's actual question? 1 = not at all, 3 = partially, 5 = fully addresses it.

Reply with only the number."""


def judge_relevancy(record):
    out = generate(RELEVANCY_PROMPT.format(question=record['question'],
                                           answer=strip_citations(record['answer'])),
                   model=JUDGE_MODEL)
    m = re.search(r'[1-5]', out)
    return {'relevancy': int(m.group()) if m else None}

In [13]:
def score(record, arm):
    """run all metrics that make sense for the arm.
    faithfulness needs passages, so the baseline arm skips it - the baseline's
    accuracy gets checked by hand in the smoke test instead."""
    scores = {}
    scores.update(readability(record['answer']))
    scores.update(judge_relevancy(record))
    if arm == 'rag':
        scores.update(citation_scores(record['answer'], record['hits']))
        scores.update(judge_faithfulness(record))
    return scores

## Calibrating the judge

Before trusting Llama's YES/NO verdicts, checking them against my own. The cell below collects
~20 sentence-level judgments into a CSV. Workflow: run it, open the CSV, fill the my_verdict
column with 1 (supported) or 0 (not) WITHOUT looking at the judge column first, then run the
agreement cell. That agreement number goes in the report - it's the answer to \"why should anyone
trust an LLM judge\"."


In [ ]:
CALIBRATION_QUESTIONS = [
    'What should I do after being discharged with heart failure?',
    'How do I take care of a surgical wound at home?',
    'What foods should I avoid with high blood sugar?',
    'What are the side effects of blood thinners?',
]

rows = []
for q in CALIBRATION_QUESTIONS:
    rec = rag_answer(q)
    passages = format_passages(rec['hits'])
    for s in split_sentences(rec['answer']):
        out = generate(FAITHFUL_PROMPT.format(passages=passages, sentence=s), model=JUDGE_MODEL)
        rows.append({
            'question': q,
            'sentence': s,
            'judge_verdict': 1 if out.strip().upper().startswith('YES') else 0,
            'my_verdict': '',      # fill by hand: 1 supported / 0 not
            'passages': passages,  # kept so I can actually check support
        })

pd.DataFrame(rows).to_csv('judge_calibration.csv', index=False)
print(f"wrote {len(rows)} sentence judgments to judge_calibration.csv - fill my_verdict, then run the next cell")

wrote 25 sentence judgments to judge_calibration.csv - fill my_verdict, then run the next cell


In [15]:
cal = pd.read_csv('judge_calibration.csv')
done = cal[cal['my_verdict'].notna() & (cal['my_verdict'].astype(str) != '')]
if len(done) == 0:
    print('my_verdict column not filled yet')
else:
    agree = (done['judge_verdict'] == done['my_verdict'].astype(int)).mean()
    print(f"judge-human agreement: {agree:.0%} on {len(done)} sentences")
    disagreements = done[done['judge_verdict'] != done['my_verdict'].astype(int)]
    for _, row in disagreements.iterrows():
        print(f"  DISAGREE (judge={row['judge_verdict']}, me={row['my_verdict']}): {row['sentence'][:90]}")

my_verdict column not filled yet


## 5 - smoke test: both arms on gold-subset questions

12 questions built around gold subset topics plus two off-corpus refusal probes. These are
template-style questions for debugging the harness - the real evaluation in week 6 uses actual
consumer questions from MedQuAD/CHQA, so no conclusions get drawn from these numbers beyond
\"does everything run and do the metrics look sane\"."


In [16]:
with open('gold_subset.json', encoding='utf-8') as f:
    gold = json.load(f)

# two question shapes per topic so it's not all one phrasing
smoke_questions = []
for i, g in enumerate(gold[:12]):
    if i % 2 == 0:
        smoke_questions.append(f"What is {g['title']} and what should I know about it?")
    else:
        smoke_questions.append(f"How is {g['title']} treated or managed?")

# off-corpus probes - the right behaviour is the refusal sentence
smoke_questions += [
    'Can you renew my car insurance for me?',
    'What is the best restaurant near the hospital?',
]

print(len(smoke_questions), 'questions')
for q in smoke_questions:
    print(' -', q)

14 questions
 - What is Tetanus and what should I know about it?
 - How is Blood Count Tests treated or managed?
 - What is Attention Deficit Hyperactivity Disorder and what should I know about it?
 - How is Infectious Arthritis treated or managed?
 - What is Heartburn and what should I know about it?
 - How is Guillain-Barre Syndrome treated or managed?
 - What is Carcinoid Tumors and what should I know about it?
 - How is Vulvar Disorders treated or managed?
 - What is COPD and what should I know about it?
 - How is Burns treated or managed?
 - What is Schizophrenia and what should I know about it?
 - How is Body Weight treated or managed?
 - Can you renew my car insurance for me?
 - What is the best restaurant near the hospital?


In [17]:
# the long cell - roughly (1 gen + 1 gen + ~10 judge calls) per question.
# budget an hour+, go make tea
results = []
for i, q in enumerate(smoke_questions):
    print(f"[{i+1}/{len(smoke_questions)}] {q[:60]}")

    rec = rag_answer(q)
    s = score(rec, 'rag')
    log_run(rec, 'rag', s)
    results.append({'arm': 'rag', 'question': q, **{k: v for k, v in s.items() if k != 'unsupported'}})

    rec = baseline_answer(q)
    s = score(rec, 'baseline')
    log_run(rec, 'baseline', s)
    results.append({'arm': 'baseline', 'question': q, **s})

df = pd.DataFrame(results)
df.to_csv('wk4_smoke_results.csv', index=False)
print('\ndone,', len(df), 'rows')

[1/14] What is Tetanus and what should I know about it?
[2/14] How is Blood Count Tests treated or managed?
[3/14] What is Attention Deficit Hyperactivity Disorder and what sh
[4/14] How is Infectious Arthritis treated or managed?
[5/14] What is Heartburn and what should I know about it?
[6/14] How is Guillain-Barre Syndrome treated or managed?
[7/14] What is Carcinoid Tumors and what should I know about it?
[8/14] How is Vulvar Disorders treated or managed?
[9/14] What is COPD and what should I know about it?
[10/14] How is Burns treated or managed?
[11/14] What is Schizophrenia and what should I know about it?
[12/14] How is Body Weight treated or managed?
[13/14] Can you renew my car insurance for me?
[14/14] What is the best restaurant near the hospital?

done, 28 rows


In [18]:
# the comparison that matters
summary = df.groupby('arm').agg(
    fkgl=('fkgl', 'mean'),
    fre=('fre', 'mean'),
    words=('words', 'mean'),
    relevancy=('relevancy', 'mean'),
    faithfulness=('faithfulness', 'mean'),
    citation_precision=('citation_precision', 'mean'),
).round(2)
print(summary)

# and the refusal probes specifically - did the RAG arm actually refuse?
print('\nrefusal probes (rag arm):')
for q in smoke_questions[-2:]:
    row = df[(df['question'] == q) & (df['arm'] == 'rag')]
    print(' -', q)

           fkgl    fre   words  relevancy  faithfulness  citation_precision
arm                                                                        
baseline  10.07  55.39  208.36       4.50           NaN                 NaN
rag        9.64  56.14  108.57       3.86          0.68                0.95

refusal probes (rag arm):
 - Can you renew my car insurance for me?
 - What is the best restaurant near the hospital?


In [20]:
# split out the two off-topic refusal probes so faithfulness isn't dragged
# down by refusal sentences (a refusal has nothing in the passages to be
# faithful to). report medical questions and refusal probes separately.
is_refusal = df['question'].str.contains('insurance|restaurant', case=False)

print("MEDICAL QUESTIONS ONLY (12):")
print(df[~is_refusal].groupby('arm').agg(
    fkgl=('fkgl', 'mean'),
    fre=('fre', 'mean'),
    words=('words', 'mean'),
    relevancy=('relevancy', 'mean'),
    faithfulness=('faithfulness', 'mean'),
    citation_precision=('citation_precision', 'mean'),
).round(2))

print("\nREFUSAL PROBES (2) — reported separately:")
print(df[is_refusal].groupby('arm').agg(
    relevancy=('relevancy', 'mean'),
    cited_any=('cited_any', 'first'),
).round(2))

MEDICAL QUESTIONS ONLY (12):
          fkgl    fre   words  relevancy  faithfulness  citation_precision
arm                                                                       
baseline  10.4  53.89  226.17       4.58           NaN                 NaN
rag        9.9  56.11  123.08       4.00          0.79                0.95

REFUSAL PROBES (2) — reported separately:
          relevancy cited_any
arm                          
baseline        4.0      None
rag             3.0     False


In [19]:
# eyeball the RAG arm's unsupported sentences across the log - these are the
# hallucination candidates to discuss in the observations
with open(RUN_LOG, encoding='utf-8') as f:
    runs = [json.loads(line) for line in f]

for r in runs:
    if r['arm'] == 'rag' and r['scores'].get('unsupported'):
        print('Q:', r['question'][:60])
        for s in r['scores']['unsupported']:
            print('   NOT SUPPORTED:', s[:100])
        print()

Q: What is Tetanus and what should I know about it?
   NOT SUPPORTED: To prevent tetanus, you should get vaccinated against it.
   NOT SUPPORTED: It's also a good idea to wear boots and long pants in areas with venomous snakes.

Q: How is Blood Count Tests treated or managed?
   NOT SUPPORTED: I don't have reliable information to tell you how blood count tests are treated or managed as they a

Q: What is Attention Deficit Hyperactivity Disorder and what sh
   NOT SUPPORTED: They can help determine if you have ADHD and provide appropriate treatment options.

Q: How is Infectious Arthritis treated or managed?
   NOT SUPPORTED: The specific treatment may depend on the cause of the infection.

Q: What is Heartburn and what should I know about it?
   NOT SUPPORTED: They may recommend tests such as X-rays or endoscopy .

Q: What is Carcinoid Tumors and what should I know about it?
   NOT SUPPORTED: In early stages, they don't produce symptoms.
   NOT SUPPORTED: However, in later stages, some